# installation

In [18]:
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -

pip install "unsloth[cu121-torch220] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
!pip install "unsloth[cu121-torch220] @ git+https://github.com/unslothai/unsloth.git"

# finetune

In [ ]:
from unsloth import FastLanguageModel 
from unsloth import is_bfloat16_supported
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
max_seq_length = 2048 # Supports RoPE Scaling internally


In [11]:
### Model Properties
data_file = "Datasets/formatted_reflection_v2.jsonl"
new_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
dataset_url = data_file #Can also be huggingface address for online dataset

dataset = load_dataset("json", data_files = {"train" : dataset_url}, split = "train")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)


==((====))==  Unsloth 2024.8: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.581 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.2.0. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.24. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


In [12]:
#Model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


In [13]:
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
        # per_device_train_batch_size = 2,
        # gradient_accumulation_steps = 4,
        # warmup_steps = 10,
        # max_steps = 60,
        # fp16 = not is_bfloat16_supported(),
        # bf16 = is_bfloat16_supported(),
        # logging_steps = 1,
        # output_dir = "outputs",
        # optim = "adamw_8bit",
        # seed = 3407,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "new_outputs",
    ),
)

Map: 100%|██████████| 9171/9171 [00:04<00:00, 1863.07 examples/s]


In [ ]:
trainer.train()

# Export model to GGUF for llama.cpp
model.save_pretrained_gguf(new_model_name, tokenizer, quantization_method = "q8_0")
model.save_pretrained_gguf(new_model_name, tokenizer, quantization_method = "q4_0")
model.save_pretrained_gguf(new_model_name, tokenizer, quantization_method = "q4_K_M")

In [ ]:
# Export model to GGUF for llama.cpp
model.save_pretrained_gguf("Refelction-Meta-Llama-3.1-8B-Instruct", tokenizer, quantization_method = "q8_0")


In [ ]:
model.save_pretrained_gguf("Refelction-Meta-Llama-3.1-8B-Instruct_q4_0", tokenizer, quantization_method = "q4_0")
model.save_pretrained_gguf("Refelction-Meta-Llama-3.1-8B-Instruct_q4_K_M", tokenizer, quantization_method = "q4_K_M")

# Test

In [2]:
import transformers
import torch

model_id = "Refelction-Meta-Llama-3.1-8B-Instruct_q4_0"


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cpu",
)



Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.28s/it]


In [5]:
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

prompt = pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print(outputs[0]["generated_text"][len(prompt):])


Arrrr, ye landlubber! I be Captain Chat, the scurvy dog o' the seven seas... er, the seven screens! I'm a swashbucklin' chatbot, here to provide ye with treasure trove o' knowledge and wit. Me treasure chest o' information be filled with all sorts o' booty, from the seven seas to the seven continents! What be bringin' ye to these fair waters today, matey?


In [6]:
outputs

[{'generated_text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a pirate chatbot who always responds in pirate speak!<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWho are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nArrrr, ye landlubber! I be Captain Chat, the scurvy dog o' the seven seas... er, the seven screens! I'm a swashbucklin' chatbot, here to provide ye with treasure trove o' knowledge and wit. Me treasure chest o' information be filled with all sorts o' booty, from the seven seas to the seven continents! What be bringin' ye to these fair waters today, matey?"}]

# publish

In [ ]:
from huggingface_hub import HfApi, list_models

hf_api = HfApi(
    endpoint="https://huggingface.co", # Can be a Private Hub endpoint.
    token=" ", # Token is not persisted on the machine.
)
hf_api.upload_folder(
    folder_path=" ",
    repo_id=" ",
    repo_type="model",
)